In [1]:
# Cài đặt các thư viện bắt buộc cho Pipeline và ép cập nhật torchao
!pip install -q -U "torchao>0.16.0"
!pip install -q open_clip_torch
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q bert_score
!pip install -q scikit-learn
!pip install evaluate rouge_score bert_score scikit-learn
!pip install -q bitsandbytes accelerate
!pip install -q -U transformers accelerate datasets
!pip install -q einops
print("✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 79.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 59.6 MB/s eta 0:00:00:00:01
✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!


In [2]:
import os
import gc
import time
import torch
import re
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from datasets import load_dataset
import transformers
from transformers import AutoModel, AutoTokenizer
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
import bitsandbytes as bnb 
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
import evaluate
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm 
import warnings
import ctypes
from peft import LoraConfig, get_peft_model

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ==========================================
# 0. VÁ LỖI KIẾN TRÚC INTERNVL & HUGGINGFACE
# ==========================================
transformers.PreTrainedModel.all_tied_weights_keys = {}
transformers.logging.set_verbosity_error()
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
torch.backends.cudnn.benchmark = True 

runName = "Centralized_VQARAD_InternVL3.5_LoRA_448_R32_Alpha64_E2"
kaggleBaseDir = "/kaggle/working/Centralized-MVQA-Project"
dataDir = os.path.join(kaggleBaseDir, "data", "vqarad")
logDir = os.path.join(kaggleBaseDir, "logs")
checkpointDir = os.path.join(kaggleBaseDir, "checkpoints", runName)

os.makedirs(dataDir, exist_ok=True)
os.makedirs(logDir, exist_ok=True)
os.makedirs(checkpointDir, exist_ok=True)

logPath = os.path.join(logDir, f"{runName}_log.csv")
checkpointPath = os.path.join(checkpointDir, "best_centralized_model_lora.pt")

# ==========================================
# 1. CÔNG CỤ QUẢN LÝ BỘ NHỚ (OOM PREVENTION)
# ==========================================
def force_garbage_collect():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    try:
        libc = ctypes.CDLL("libc.so.6")
        libc.malloc_trim(0)
    except Exception:
        pass

def getPeakReservedMemory():
    return f"{torch.cuda.memory_reserved() / (1024 ** 3):.2f} GB" if torch.cuda.is_available() else "0.00 GB"

def processImage(pilImg, targetSize=448):
    if pilImg.mode != "RGB": 
        pilImg = pilImg.convert("RGB")
    imageTransform = T.Compose([
        T.Resize((targetSize, targetSize), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    ])
    return imageTransform(pilImg).unsqueeze(0)

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH TOÀN CỤC & TIÊM LORA (r=32, alpha=64)
# ==========================================
print("--- KHỞI TẠO HẠ TẦNG CENTRALIZED-LORA (INTERNVL3.5-2B, r=32, alpha=64) ---")
force_garbage_collect()

modelId = "OpenGVLab/InternVL3_5-2B-Instruct"
baseModel = AutoModel.from_pretrained(
    modelId,
    torch_dtype=torch.float16, 
    trust_remote_code=True,
    low_cpu_mem_usage=True 
).cuda()

tokenizer = AutoTokenizer.from_pretrained(modelId, trust_remote_code=True)
print("✅ Đã nạp thành công mô hình Base InternVL3.5-2B!")

baseModel.config.use_cache = False
if hasattr(baseModel, 'enable_input_require_grads'):
    baseModel.enable_input_require_grads()

try:
    baseModel.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except: pass

if hasattr(baseModel, 'language_model'):
    baseModel.language_model.gradient_checkpointing_enable()
if hasattr(baseModel, 'vision_model'):
    baseModel.vision_model.gradient_checkpointing_enable()

for p in baseModel.parameters():
    p.requires_grad = False

# CẤU HÌNH LORA THEO YÊU CẦU: r=32, lora_alpha=64
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none"
)

baseModel = get_peft_model(baseModel, lora_config)
print("✅ Đã tiêm cấu trúc LoRA thành công!")

trainableParams = sum(p.numel() for p in baseModel.parameters() if p.requires_grad)
totalModelParams = sum(p.numel() for p in baseModel.parameters())
print(f"[Metrics] Trainable Params (LoRA): {trainableParams:,} ({100 * trainableParams / totalModelParams:.2f}%)")
force_garbage_collect()

# ==========================================
# 3. TRÍCH XUẤT ĐỘNG ID TOKEN HÌNH ẢNH 
# ==========================================
print("Đang trích xuất tự động Image Context ID từ lõi mô hình...")
interceptedIds = None
originalForward = baseModel.forward

def hookForward(*args, **kwargs):
    global interceptedIds
    if 'input_ids' in kwargs and kwargs['input_ids'] is not None:
        interceptedIds = kwargs['input_ids'].detach().cpu()
    elif len(args) > 1 and args[1] is not None:
        interceptedIds = args[1].detach().cpu()
    raise RuntimeError("STOP_INTERCEPT")

baseModel.forward = hookForward
try:
    dummyPx = torch.zeros((1, 3, 448, 448), dtype=torch.float16).cuda()
    baseModel.chat(tokenizer, dummyPx, "", generation_config=dict(max_new_tokens=1))
except RuntimeError as e:
    if str(e) != "STOP_INTERCEPT": pass
except Exception: pass
finally:
    baseModel.forward = originalForward

if interceptedIds is not None:
    counts = torch.bincount(interceptedIds.view(-1))
    imgContextId = int(counts.argmax().item())
else:
    imgContextId = getattr(baseModel, 'img_context_token_id', tokenizer.convert_tokens_to_ids(''))
    
print(f"✅ Đã tìm thấy Image Context ID an toàn: {imgContextId}")

# ==========================================
# 4. CHUẨN BỊ DỮ LIỆU TẬP TRUNG VQA-RAD
# ==========================================
print("\n--- CHUẨN BỊ DỮ LIỆU TẬP TRUNG VQA-RAD ---")

dataset = load_dataset("flaviagiammarino/vqa-rad", cache_dir=dataDir)
trainDataset = dataset["train"].shuffle(seed=42)
testDataset = dataset["test"]

print(f"Dữ liệu Train tập trung: {len(trainDataset)} mẫu")
print(f"Dữ liệu Test tập trung : {len(testDataset)} mẫu")

systemInstruction = (
    "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
    "based EXCLUSIVELY on the provided medical image.\nStrictly follow these rules:\n"
    "1. Analyze the provided image.\n"
    "2. Provide a concise and definitive answer (1-3 words if possible)."
)

def customCollateFn(batch):
    pixelValuesList, inputIdsList, labelsList = [], [], []
    for item in batch:
        img = item["image"].convert("RGB") 
        q, a = item["question"], str(item["answer"]).strip()
        
        promptPart1 = f"{systemInstruction}\nUser: "
        promptPart2 = f"\nQuestion: {q}\nFinal Answer:\nAssistant: "

        part1Ids = tokenizer(promptPart1, return_tensors="pt", add_special_tokens=True).input_ids[0]
        part2Ids = tokenizer(promptPart2, return_tensors="pt", add_special_tokens=False).input_ids[0]
        answerIds = tokenizer(a + getattr(tokenizer, 'eos_token', ''), return_tensors="pt", add_special_tokens=False).input_ids[0]
        
        imgTokens = torch.full((256,), imgContextId, dtype=torch.long)
        inputIds = torch.cat([part1Ids, imgTokens, part2Ids, answerIds], dim=0)
        
        lbls = inputIds.clone()
        promptLen = len(part1Ids) + 256 + len(part2Ids)
        lbls[:promptLen] = -100 
        
        inputIdsList.append(inputIds)
        labelsList.append(lbls)
        pixelValuesList.append(processImage(img))

    maxLen = max(len(ids) for ids in inputIdsList)
    padId = getattr(tokenizer, 'pad_token_id', getattr(tokenizer, 'eos_token_id', 0))
    if padId is None: 
        padId = 0

    paddedInputIds, paddedLabels, attentionMask = [], [], []
    for ids, lbls in zip(inputIdsList, labelsList):
        padLen = maxLen - len(ids)
        paddedInputIds.append(torch.cat([ids, torch.full((padLen,), padId, dtype=torch.long)]))
        paddedLabels.append(torch.cat([lbls, torch.full((padLen,), -100, dtype=torch.long)]))
        attentionMask.append(torch.cat([torch.ones(len(ids), dtype=torch.long), torch.zeros(padLen, dtype=torch.long)]))

    return {
        "input_ids": torch.stack(paddedInputIds),
        "labels": torch.stack(paddedLabels),
        "attention_mask": torch.stack(attentionMask),
        "pixel_values": torch.cat(pixelValuesList, dim=0).to(torch.float16),
        "image_flags": torch.ones((len(batch), 1), dtype=torch.long) 
    }

batchSize = 1
trainLoader = DataLoader(
    trainDataset,
    batch_size=batchSize,
    shuffle=True,
    collate_fn=customCollateFn,
    num_workers=0,
    pin_memory=False
)

# ==========================================
# 5. HÀM ĐÁNH GIÁ
# ==========================================
def evaluateModel(model, datasetInput):
    model.eval()
    print(f"\n [Server] Đang đánh giá trên tập Test ({len(datasetInput)} mẫu)...")

    openPreds, openRefs, closePreds, closeRefs, inferenceTimes = [], [], [], [], []
    evalLoop = tqdm(datasetInput, desc="Evaluating", leave=False, dynamic_ncols=True)

    for item in evalLoop:
        q, gtAnswer = item["question"], str(item["answer"]).strip().lower()
        ansType = str(item.get("answer_type", "CLOSED" if gtAnswer in ["yes", "no"] else "OPEN")).strip().upper()
        
        promptText = f"{systemInstruction}\n[Target Case Image]\n\n\nCurrent Question: {q}\nBased on the visual evidence, what is the correct answer?\nFinal Answer: "
        img = item["image"].convert("RGB")
        pixelValues = processImage(img).to(torch.float16).cuda()
        
        startTime = time.time()
        with torch.inference_mode():
            aiAnswerRaw = model.chat(
                tokenizer=tokenizer,
                pixel_values=pixelValues,
                question=promptText,
                generation_config=dict(max_new_tokens=15, do_sample=False)
            )
        inferenceTimes.append(time.time() - startTime)
        aiAnswer = aiAnswerRaw.lower().strip()
        
        if ansType == "CLOSED":
            predClean = 'yes' if 'yes' in aiAnswer and 'no' not in aiAnswer else ('no' if 'no' in aiAnswer and 'yes' not in aiAnswer else ('yes' if aiAnswer.find('yes') < aiAnswer.find('no') else 'no'))
            closePreds.append(predClean)
            closeRefs.append('yes' if 'yes' in gtAnswer else ('no' if 'no' in gtAnswer else gtAnswer))
        else:
            cleanStr = lambda t: " ".join([w for w in re.sub(r'[^\w\s]', '', t.lower()).split() if w not in {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'it', 'there', 'shows', 'showing', 'of', 'in', 'on', 'to'}])
            openPreds.append(cleanStr(aiAnswer) if cleanStr(aiAnswer) else "none")
            openRefs.append(cleanStr(gtAnswer) if cleanStr(gtAnswer) else "none")
            
        del pixelValues
        torch.cuda.empty_cache()
        
    res = {"avgInferenceTime": np.mean(inferenceTimes), "overallAccuracy": (sum(p==r for p, r in zip(closePreds, closeRefs)) + sum(p==r for p, r in zip(openPreds, openRefs))) / len(datasetInput)}
    if closeRefs: 
        res["Closed"] = {"ACC": accuracy_score(closeRefs, closePreds), "F1": f1_score(closeRefs, closePreds, average='weighted', zero_division=0), "count": len(closeRefs)}
    if openRefs:
        bertRes = evaluate.load("bertscore").compute(predictions=openPreds, references=openRefs, lang="en")
        res["Open"] = {"BLEU-1": evaluate.load("bleu").compute(predictions=openPreds, references=[[r] for r in openRefs], max_order=1)["bleu"], "ROUGE-L": evaluate.load("rouge").compute(predictions=openPreds, references=[[r] for r in openRefs])["rougeL"], "BERTScore": np.mean(bertRes["f1"]), "count": len(openRefs)}
    return res

# ==========================================
# 6. VÒNG LẶP HUẤN LUYỆN TẬP TRUNG (SỬA LẠI LOGIC TÍNH LOSS CHUẨN XÁC)
# ==========================================
epochs = 2
learningRate = 2e-4
accumulationSteps = 4 

optimizer = bnb.optim.PagedAdamW8bit(baseModel.parameters(), lr=learningRate, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs * len(trainLoader) // accumulationSteps)

print(f"\n=== BẮT ĐẦU HUẤN LUYỆN TẬP TRUNG INTERNVL3.5-2B (VQA-RAD, {epochs} EPOCHS) ===")
baseModel.train()
startTime = time.time()
epochLogs = []

for epoch in range(epochs):
    totalLoss = 0.0
    optimizer.zero_grad(set_to_none=True)
    progressLoop = tqdm(trainLoader, desc=f"Epoch {epoch+1}/{epochs}", leave=True, dynamic_ncols=True)
    
    for step, batch in enumerate(progressLoop):
        batch = {k: v.cuda(non_blocking=True) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        
        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = baseModel(**batch)
            loss = outputs.loss
            scaledLoss = loss / accumulationSteps
        
        scaledLoss.backward()
        
        # Lưu lại giá trị loss gốc (chưa chia) để cộng dồn chính xác
        lossVal = loss.item()
        totalLoss += lossVal
        
        if (step + 1) % accumulationSteps == 0 or (step + 1) == len(trainLoader):
            torch.nn.utils.clip_grad_norm_(baseModel.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            
        progressLoop.set_postfix(Loss=f"{lossVal:.4f}")
        
        del outputs, loss, scaledLoss, batch
        if step % 20 == 0:  
            force_garbage_collect()
            
    avgLoss = totalLoss / len(trainLoader)
    epochLogs.append({"Epoch": epoch + 1, "Avg Loss": round(avgLoss, 4)})
    print(f"✅ Hoàn thành Epoch {epoch+1}/{epochs} | Avg Loss: {avgLoss:.4f}")

trainDuration = time.time() - startTime
peakVram = getPeakReservedMemory()

torch.save(baseModel.state_dict(), checkpointPath)
print(f"\n -> Đã lưu checkpoint tại: {checkpointPath}")

# ==========================================
# 7. ĐÁNH GIÁ MÔ HÌNH VÀ BÁO CÁO KẾT QUẢ
# ==========================================
baseModel.config.use_cache = True
evalRes = evaluateModel(baseModel, testDataset)
baseModel.config.use_cache = False

print("\n" + "="*80)
print(f" FINAL SUMMARY REPORT: CENTRALIZED INTERNVL3.5 (VQA-RAD, {epochs} EPOCHS) ")
print("="*80)

print("\n[1] PERFORMANCE & HARDWARE METRICS")
print(f"  + Trainable Parameters   : {trainableParams:,} ({100 * trainableParams / totalModelParams:.2f}%)")
print(f"  + Peak Reserved Memory   : {peakVram}")
print(f"  + Total Training Time    : {trainDuration:.2f} giây")
print(f"  + Average Inference Time : {evalRes['avgInferenceTime']:.4f} giây/ảnh")

print("\n[2] CLINICAL EVALUATION METRICS")
print(f"  - OVERALL ACCURACY       : {evalRes['overallAccuracy']*100:.2f}%")
if "Closed" in evalRes:
    print(f"  - CLOSED QUESTIONS ({evalRes['Closed'].get('count', 'N/A')} samples):")
    print(f"    * Accuracy : {evalRes['Closed']['ACC']*100:.2f}%")
    print(f"    * F1-Score : {evalRes['Closed']['F1']*100:.2f}%")
if "Open" in evalRes:
    print(f"  - OPEN QUESTIONS ({evalRes['Open'].get('count', 'N/A')} samples):")
    print(f"    * BLEU-1   : {evalRes['Open']['BLEU-1']:.4f}")
    print(f"    * ROUGE-L  : {evalRes['Open']['ROUGE-L']:.4f}")
    print(f"    * BERTScore: {evalRes['Open']['BERTScore']:.4f}")

print("\n[3] TRAINING LOSS CONVERGENCE")
lossTable = pd.DataFrame(epochLogs)
print(lossTable.to_string(index=False))
lossTable.to_csv(logPath, index=False)

print("\n" + "="*80)
print(" QUÁ TRÌNH THỰC NGHIỆM ĐÃ HOÀN TẤT THÀNH CÔNG ")
print("="*80)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


--- KHỞI TẠO HẠ TẦNG CENTRALIZED-LORA (INTERNVL3.5-2B, r=32, alpha=64) ---


config.json: 0.00B [00:00, ?B/s]

configuration_internvl_chat.py: 0.00B [00:00, ?B/s]

configuration_intern_vit.py: 0.00B [00:00, ?B/s]

modeling_internvl_chat.py: 0.00B [00:00, ?B/s]

... (27 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


 FINAL SUMMARY REPORT: CENTRALIZED INTERNVL3.5 (VQA-RAD, 2 EPOCHS) 

[1] PERFORMANCE & HARDWARE METRICS
  + Trainable Parameters   : 34,865,152 (1.46%)
  + Peak Reserved Memory   : 6.32 GB
  + Total Training Time    : 3151.59 giây
  + Average Inference Time : 0.3535 giây/ảnh

[2] CLINICAL EVALUATION METRICS
  - OVERALL ACCURACY       : 52.77%
  - CLOSED QUESTIONS (251 samples):
    * Accuracy : 72.91%
    * F1-Score : 72.93%
  - OPEN QUESTIONS (200 samples):
    * BLEU-1   : 0.2701
    * ROUGE-L  : 0.3637
    * BERTScore: 0.9052

[3] TRAINING LOSS CONVERGENCE
 Epoch  Avg Loss
     1    0.8376
     2    0.3335

 QUÁ TRÌNH THỰC NGHIỆM ĐÃ HOÀN TẤT THÀNH CÔNG 
